# Q/M tool for stable trapping

Estimates the window of $Q/M$ where a particle is stably trapped, for a given drive, endcap voltage and geometry.
All the inputs are in the **Parameters** cell: plug in the values there and run everything.

- The efficiency factors are set to typical blade-trap values ($\kappa = 0.473$, $\eta = 0.617$, from the fits in `notebook_extract`). Both can be changed with the sliders (1 is the ideal quadrupole field).
  With $r_0 \approx z_0$ and one pair grounded the endcaps matter a lot, so the result is sensitive to $\kappa$.
- Gravity is taken along $-y$ (towards the grounded pair, the weakest radial axis for the grounded drive), the worst case until we know the orientation of the trap.
- Damping is not included: the limits are valid at low pressure ($\lesssim 1$ mbar), not at atmospheric pressure.

## Imports

In [1]:
using MathieuFunctions,
DataFrames,
NonlinearSolve,
Printf,
NaNMath

using Makie; using Makie: Axis
import GLMakie, CairoMakie
CairoMakie.activate!()

## Functions

### General

In [2]:
function getMathieuParams(Q╱M, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=:symmetric)
    a_x = -(8*Q╱M*V_EC*κ*ϵ)/(Ω^2*z_0^2)
    a_y = a_x*(1/ϵ-1)
    a_z = -a_x/ϵ

    q_x = -(2*Q╱M*V_RF*η)/(Ω^2*r_0^2)
    q_y = -q_x
    q_z = 0.0

    # One pair grounded: the extra V_RF/2 on all the blades acts like -V_RF/2 (AC) on the endcaps
    if drive == :grounded
        q_c = (2*Q╱M*V_RF*κ)/(Ω^2*z_0^2)
        q_x -= ϵ*q_c
        q_y -= (1-ϵ)*q_c
        q_z += q_c
    end

    return a_x, a_y, a_z, q_x, q_y, q_z
end

function getSecularFreqs(Q╱M, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=:symmetric)
    a_x, a_y, a_z, q_x, q_y, q_z = getMathieuParams(Q╱M, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)

    # Adiabatic approximation (good for q ≲ 0.4)
    ω_x = Ω/2*NaNMath.sqrt(a_x + q_x^2/2)
    ω_y = Ω/2*NaNMath.sqrt(a_y + q_y^2/2)
    ω_z = Ω/2*NaNMath.sqrt(a_z + q_z^2/2)

    return ω_x, ω_y, ω_z
end

function getMathieuEdges(s)
    # Edges of the first stability region a₀(q) < a < b₁(q) along the line a = s*q
    edge_lo(q, p) = charA(q, k=0:0)[1] - p.s*q
    edge_hi(q, p) = charB(q, k=1:1)[1] - p.s*q
    p = (s=s,)

    if s >= 0
        q_lo = 0.0
    elseif edge_lo(1e-9, p) > 0 && edge_lo(0.908, p) < 0
        q_lo = solve(IntervalNonlinearProblem(edge_lo, (1e-9, 0.908), p), ITP()).u
    else
        return NaN, NaN # the line never enters the stability region
    end
    q_hi = solve(IntervalNonlinearProblem(edge_hi, (max(q_lo, 1e-9), 1.5), p), ITP()).u

    return q_lo, q_hi
end

function getQ╱MEdges(V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=:symmetric)
    # a_i and q_i are ∝ Q/M, so evaluate them at Q/M = 1 and scale
    a_x, a_y, a_z, q_x, q_y, q_z = getMathieuParams(1.0, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)

    # The window is where the three axes are stable
    Q╱M_lo, Q╱M_hi = 0.0, Inf
    for (a, q) in ((a_x, q_x), (a_y, q_y), (a_z, q_z))
        if q == 0
            a > 0 || return NaN, NaN # only static potential, needs a > 0
            continue
        end
        q_lo, q_hi = getMathieuEdges(a/abs(q))
        Q╱M_lo = max(Q╱M_lo, q_lo/abs(q))
        Q╱M_hi = min(Q╱M_hi, q_hi/abs(q))
    end

    return Q╱M_lo, Q╱M_hi
end

function getQ╱MSag(V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; sag_max=0.1*r_0, g=9.81, drive=:symmetric)
    # a_y and q_y at Q/M = 1
    _, a_y, _, _, q_y, _ = getMathieuParams(1.0, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)

    # Gravity along y: g/ω_y² = sag_max  →  q_y²/2*(Q/M)² + a_y*(Q/M) - 4g/(Ω²*sag_max) = 0
    return (-a_y + sqrt(a_y^2 + 8*q_y^2*g/(Ω^2*sag_max))) / q_y^2
end

function getQ╱MWindow(V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; q_ad=0.4, sag_max=0.1*r_0, g=9.81, drive=:symmetric)
    Q╱M_lo, Q╱M_hi = getQ╱MEdges(V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)
    q_max = maximum(abs.(getMathieuParams(1.0, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)[4:6])) # max |q_i| at Q/M = 1

    return (
        Q╱M_lo  = Q╱M_lo,     # Mathieu, lower edge (a₀)
        Q╱M_sag = getQ╱MSag(V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; sag_max=sag_max, g=g, drive=drive),
        Q╱M_ad  = q_ad/q_max, # Adiabatic limit
        Q╱M_hi  = Q╱M_hi,     # Mathieu, upper edge (b₁)
    )
end

function getStabilityMaps(x_var, xs, Q╱Ms, p, V_EC, η, κ, r_0, z_0, ϵ; q_ad=0.4, sag_max=0.1*r_0, g=9.81, drive=:symmetric)
    # x_var = :f    → xs are frequencies and p = V_RF
    # x_var = :V_RF → xs are RF amplitudes and p = f
    n_x = length(xs)
    qs  = fill(NaN, n_x, length(Q╱Ms))
    f_y = fill(NaN, n_x, length(Q╱Ms))
    Q╱M_ad  = zeros(n_x)
    Q╱M_sag = zeros(n_x)

    # The Mathieu and adiabatic limits scale as Ω², so for the f map they are only solved once (gravity doesn't)
    if x_var == :f
        win = getQ╱MWindow(p, V_EC, 2π*xs[1], η, κ, r_0, z_0, ϵ; q_ad=q_ad, sag_max=sag_max, g=g, drive=drive)
        wins = [merge(map(Q╱M -> Q╱M*(x/xs[1])^2, win),
                      (Q╱M_sag = getQ╱MSag(p, V_EC, 2π*x, η, κ, r_0, z_0, ϵ; sag_max=sag_max, g=g, drive=drive),)) for x in xs]
    else
        wins = [getQ╱MWindow(x, V_EC, 2π*p, η, κ, r_0, z_0, ϵ; q_ad=q_ad, sag_max=sag_max, g=g, drive=drive) for x in xs]
    end

    for (i, x) in enumerate(xs)
        V_RF, f = x_var == :f ? (p, x) : (x, p)
        Ω = 2π*f
        for (j, Q╱M) in enumerate(Q╱Ms)
            if wins[i].Q╱M_lo <= Q╱M <= wins[i].Q╱M_hi
                qs[i, j]  = abs(getMathieuParams(Q╱M, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)[4])
                f_y[i, j] = getSecularFreqs(Q╱M, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)[2] / 2π
            end
        end
        Q╱M_ad[i]  = wins[i].Q╱M_ad
        Q╱M_sag[i] = wins[i].Q╱M_sag
    end

    return qs, f_y, Q╱M_ad, Q╱M_sag
end

getStabilityMaps (generic function with 1 method)

### Graphics

In [3]:
function plotQ╱MMap(x_var, x_range, Q╱M_range, η, κ, r_0, z_0, ϵ, V_EC, V_RF, f, N, M;
            drive=:symmetric,
            e⁻=1.60217663e-19,
            colormap=:viridis,
            res=250,
            interactive=true,
            q_ad=0.4,
            sag_max=0.1*r_0,
            g=9.81,
            ω_levels=[10, 20, 50, 100, 200, 500, 1000, 2000, 5000],
            legend_position=(x_var == :f ? :lt : :rt),
            V_RF_range=(10:1:1000),
            f_range=(0.5:0.1:100),
            V_EC_range=(-10:0.1:10),
            r_0_range=(0.5:0.001:2),
            z_0_range=(0.5:0.001:5),
            η_range=(0.01:0.001:1),
            κ_range=(0.01:0.001:1),
            sag_range=(10:10:1000),
            N_range=sort(unique([N; round.(Int, 10 .^ LinRange(0, log10(max(Q╱M_range[2]*M/e⁻, 2)), 1000))]))
            )
    GLMakie.activate!()
    @assert x_var in (:f, :V_RF) "x_var must be :f or :V_RF"
    xs   = x_var == :f ? 10 .^ LinRange(log10(x_range[1]), log10(x_range[2]), res) : LinRange(x_range[1], x_range[2], res)
    Q╱Ms = 10 .^ LinRange(log10(Q╱M_range[1]), log10(Q╱M_range[2]), res)
    x_unit = x_var == :f ? 1e-3 : 1.0 # f in kHz, V_RF in V

    # Observables
    ## Sliders
    V_RF_obs = Observable(V_RF)
    f_obs    = Observable(f)
    V_EC_obs = Observable(V_EC)
    r_0_obs  = Observable(r_0)
    z_0_obs  = Observable(z_0)
    η_obs    = Observable(η)
    κ_obs    = Observable(κ)
    sag_obs  = Observable(sag_max)
    N_obs    = Observable(N)

    ## Parameter that is not on the x axis (the maps don't depend on the cursor)
    p_obs = x_var == :f ? V_RF_obs : f_obs

    ## Frequency maps of q_x and ω_y in the stable region, adiabatic and gravity limits
    maps_obs = @lift getStabilityMaps(x_var, xs, Q╱Ms, $p_obs, $V_EC_obs, $η_obs, $κ_obs, $r_0_obs, $z_0_obs, ϵ;
                                      q_ad=q_ad, sag_max=$sag_obs, g=g, drive=drive)
    qs_obs      = @lift $maps_obs[1]
    f_y_obs     = @lift $maps_obs[2]
    Q╱M_ad_obs  = @lift $maps_obs[3]
    Q╱M_sag_obs = @lift $maps_obs[4]

    ## Current operating point info
    ### Label
    Q╱M_obs = @lift $N_obs*e⁻/M
    info_obs = @lift begin
        Ω_here = 2π*$f_obs
        a_x, a_y, a_z, q_x, q_y, q_z = getMathieuParams($Q╱M_obs, $V_RF_obs, $V_EC_obs, Ω_here, $η_obs, $κ_obs, $r_0_obs, $z_0_obs, ϵ; drive=drive)
        ω_x, ω_y, ω_z = getSecularFreqs($Q╱M_obs, $V_RF_obs, $V_EC_obs, Ω_here, $η_obs, $κ_obs, $r_0_obs, $z_0_obs, ϵ; drive=drive)
        Q╱M_lo, Q╱M_hi = getQ╱MEdges($V_RF_obs, $V_EC_obs, Ω_here, $η_obs, $κ_obs, $r_0_obs, $z_0_obs, ϵ; drive=drive)
        stable = Q╱M_lo <= $Q╱M_obs <= Q╱M_hi
        @sprintf("Current Operating Point:
                \nV_RF = %.1f V
                \nΩ/2π = %.2f kHz
                \nN = %d
                \nQ/M = %.3e C/kg
                \naₓ = %.2e,  a_z = %.2e
                \nqₓ = %.3f,  q_y = %.3f,  q_z = %.3f
                \nωₓ/2π = %.1f Hz
                \nω_y/2π = %.1f Hz
                \nω_z/2π = %.1f Hz
                \nsag = %.1f µm
                \nStable: %s",
                $V_RF_obs, $f_obs*1e-3, $N_obs, $Q╱M_obs,
                a_x, a_z, q_x, q_y, q_z,
                ω_x/2π, ω_y/2π, ω_z/2π, g/ω_y^2*1e6, stable ? "yes" : "no")
    end

    ### Draggable point (x from dragging, Q/M from the N slider)
    x_here = x_var == :f ? f : V_RF
    cursor = Observable([Point2f(x_here*x_unit, N*e⁻/M)])
    cursor_color = Observable(:black)

    # Figure
    fig = Figure(size=(1000, 700))
    ax = Axis(fig[1, 1],
        title=(@lift "Q/M Map (η=$(round($η_obs, digits=3)), κ=$(round($κ_obs, digits=3)), ϵ=$(round(ϵ, digits=3)), $(drive) drive)"), titlesize=15,
        xlabel=(x_var == :f ? L"\Omega/2\pi\,\, \mathrm{(kHz)}" : L"V_{RF}\,\, \mathrm{(V)}"), xlabelsize=20,
        ylabel=L"Q/M\,\, \mathrm{(C/kg)}", ylabelsize=20,
        xscale=(x_var == :f ? log10 : identity), yscale=log10,
        backgroundcolor=:gray80
    )

    # Contour plot of q_x
    hm = contourf!(ax, xs.*x_unit, Q╱Ms, qs_obs; colormap=colormap, levels=0:0.1:0.9)
    Colorbar(fig[1, 2], hm; label=L"|q_x|", labelsize=20, ticks=0:0.1:1)

    # Isocurves of ω_y (weakest radial axis for the grounded drive)
    contour!(ax, xs.*x_unit, Q╱Ms, f_y_obs; levels=ω_levels, color=:white, labels=true,
             linestyle=:dash, linewidth=1.5, labelfont=:bold, labelsize=15)

    # Adiabatic limit and gravity limit
    lines!(ax, xs.*x_unit, Q╱M_ad_obs;  color=:red, linewidth=2, linestyle=:dash)
    lines!(ax, xs.*x_unit, Q╱M_sag_obs; color=:orange, linewidth=2.5)

    # Current operating point
    sc = scatter!(ax, cursor, color=cursor_color, markersize=15, strokecolor=:white, strokewidth=1.5)

    # Set limits and legend
    xlims!(ax, x_range[1]*x_unit, x_range[2]*x_unit)
    ylims!(ax, Q╱M_range[1], Q╱M_range[2])
    axislegend(ax,
        [LineElement(color=:white, linestyle=:dash, linewidth=1.5),
         LineElement(color=:red, linestyle=:dash, linewidth=2),
         LineElement(color=:orange, linewidth=2.5),
         MarkerElement(color=:black, marker=:circle, markersize=12),
         PolyElement(color=:gray80)],
        ["ω_y/2π (Hz)",
         "Adiabatic Limit (q = $(q_ad))",
         "Gravity Limit (stable above)",
         "Current Point",
         "Unstable"];
        position=legend_position, backgroundcolor=:gray85, framecolor=(:gray85, 0.5))

    # Interactive
    if interactive
        resize!(fig.scene, (3000, 2000))

        # Right panel for sliders and info
        right_panel = GridLayout(fig[1, 3]; tellheight=false)
        Label(right_panel[1, 1], info_obs;
            tellwidth=false,
            fontsize=15,
            font=:bold,
            halign=:center, valign=:center,
            color=:gray30,
            padding=(10, 0, 8, 8),
        )

        # Sliders
        p_slider = x_var == :f ?
            (label = "V_RF", range = V_RF_range, format = "{:.0f} V", startvalue = V_RF) :
            (label = "Ω/2π", range = f_range, format = "{:.1f} kHz", startvalue = f*1e-3)
        sg = SliderGrid(
            right_panel[2, 1],
            p_slider,
            (label = "V_EC", range = V_EC_range, format = "{:.1f} V", startvalue = V_EC),
            (label = "r_0", range = r_0_range, format = "{:.3f} mm", startvalue = r_0*1e3),
            (label = "z_0", range = z_0_range, format = "{:.3f} mm", startvalue = z_0*1e3),
            (label = "η", range = η_range, format = "{:.3f}", startvalue = η),
            (label = "κ", range = κ_range, format = "{:.3f}", startvalue = κ),
            (label = "Max. sag", range = sag_range, format = "{:.0f} µm", startvalue = sag_max*1e6),
            (label = "N", range = N_range, format = "{:d} e", startvalue = N),
            width = 350,
            tellheight = false
        )
        on(sg.sliders[1].value) do val
            if x_var == :f
                V_RF_obs[] = val
            else
                f_obs[] = val * 1e3      # kHz -> Hz
            end
        end
        on(sg.sliders[2].value) do val
            V_EC_obs[] = val
        end
        on(sg.sliders[3].value) do val
            r_0_obs[] = val * 1e-3   # mm -> m
        end
        on(sg.sliders[4].value) do val
            z_0_obs[] = val * 1e-3   # mm -> m
        end
        on(sg.sliders[5].value) do val
            η_obs[] = val
        end
        on(sg.sliders[6].value) do val
            κ_obs[] = val
        end
        on(sg.sliders[7].value) do val
            sag_obs[] = val * 1e-6   # µm -> m
        end
        on(sg.sliders[8].value) do val
            N_obs[] = val
            cursor[][1] = Point2f(cursor[][1][1], Q╱M_obs[])
            notify(cursor)
        end

        # Draggable point to adjust the x of the current operating point
        dragging = Ref(false)
        idx = Ref(1)
        on(events(fig).mousebutton, priority=2) do event
            if event.button == Mouse.left
                if event.action == Mouse.press
                    plt, i = pick(fig)
                    dragging[] = plt == sc
                    idx[] = i
                    cursor_color[] = dragging[] ? :gray40 : :black
                    return Consume(dragging[])
                elseif event.action == Mouse.release
                    dragging[] = false
                    cursor_color[] = :black
                    return Consume(false)
                end
            end
            return Consume(false)
        end
        on(events(fig).mouseposition, priority=2) do mp
            if dragging[]
                # mouseposition is in log space for log axes
                pt = Makie.apply_transform(Makie.inverse_transform(ax.scene.transformation.transform_func[]), mouseposition(ax))
                cursor[][idx[]] = Point2f(pt[1], Q╱M_obs[])
                notify(cursor)

                if x_var == :f
                    f_obs[] = pt[1] * 1e3
                else
                    V_RF_obs[] = pt[1]
                end
                return Consume(true)
            end
            return Consume(false)
        end

        # Button to reset sliders
        btt = Button(right_panel[3, 1]; label="Reset", tellwidth=false)
        on(btt.clicks) do _
            set_close_to!(sg.sliders[1], x_var == :f ? V_RF : f*1e-3)
            set_close_to!(sg.sliders[2], V_EC)
            set_close_to!(sg.sliders[3], r_0*1e3)
            set_close_to!(sg.sliders[4], z_0*1e3)
            set_close_to!(sg.sliders[5], η)
            set_close_to!(sg.sliders[6], κ)
            set_close_to!(sg.sliders[7], sag_max*1e6)
            set_close_to!(sg.sliders[8], N)
            V_RF_obs[] = V_RF
            f_obs[] = f
            cursor[][1] = Point2f(x_here*x_unit, Q╱M_obs[])
            notify(cursor)
        end

        # Show fig
        display(GLMakie.Screen(), fig.scene)
    else
        CairoMakie.activate!()
        display(fig)
    end
    return fig, ax
end

plotQ╱MMap (generic function with 1 method)

## Parameters

In [8]:
# Drive and endcaps
drive = :grounded # :grounded (one pair AC, the other GND) or :symmetric (±V_RF/2 on the two pairs)
V_RF = 200.0 # AC amplitude [V] (±200 V on the AC blades → 200 V)
f = 5.1e3 # AC frequency [Hz] (fixed)
Ω = 2*π*f
V_EC = 5.0 # Endcap DC voltage [V], same sign as the particle charge

# Trap geometry
r_0 = 0.998e-3 # Center to blade [m]
z_0 = 1e-3 # Center to endcap [m]
ϵ = 1/2 # Split of the endcap field between x and y (1/2 for a symmetric trap)

# Efficiency factors, typical blade trap (fits in notebook_extract), 1 = ideal quadrupole fields
κ = 0.473
η = 0.617

# Particle, only for the charge number N
e⁻ = 1.60217663e-19 # Elementary charge [C]
ρ = 2200 # Density [kg/m³]
d = 300e-9 # Diameter [m] (300, 500 or 1000 nm)
M = ρ*4/3*π*(d/2)^3 # Mass [kg]
N = 250 # Starting charge number

# Criteria for stable trapping
g = 9.81 # Gravitational acceleration [m/s²] (along -y)
q_ad = 0.4 # Adiabatic limit
sag_max = 0.1*r_0 # Max. gravitational sag [m]

# Map ranges (axes) and slider ranges
f_lims    = (1e3, 20e3) # Ω/2π [Hz]
V_RF_lims = (50, 250) # V_RF [V]
V_EC_lims = (-10, 10) # V_EC [V]
Q╱M_lims  = (1e-2, 1e2) # Q/M [C/kg]

(0.01, 100.0)

## Q/M window

### At the current operating point

In [5]:
win = getQ╱MWindow(V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; q_ad=q_ad, sag_max=sag_max, g=g, drive=drive)
Q╱Ms = [win.Q╱M_lo, win.Q╱M_sag, win.Q╱M_ad, win.Q╱M_hi]

display(
    DataFrame(
    "Limit"      => ["Mathieu, lower edge (a₀)", "Gravity, sag = $(round(Int, sag_max*1e6)) µm", "Adiabatic, max|q| = $(q_ad)", "Mathieu, upper edge (b₁)"],
    "Q/M (C/kg)" => Q╱Ms,
    "|q_x|"      => [abs(getMathieuParams(Q╱M_i, V_RF, V_EC, Ω, η, κ, r_0, z_0, ϵ; drive=drive)[4]) for Q╱M_i in Q╱Ms]
    )
)

Q╱M_min = max(win.Q╱M_lo, win.Q╱M_sag)
Q╱M_max = min(win.Q╱M_ad, win.Q╱M_hi)
if !(Q╱M_min < win.Q╱M_hi)
    println("No stable Q/M for these parameters")
elseif Q╱M_min < Q╱M_max
    @printf("Recommended Q/M: %.2e - %.2e C/kg\n", Q╱M_min, Q╱M_max)
else
    @printf("Stable Q/M: %.2e - %.2e C/kg, all above the adiabatic limit (q = %.1f)\n", Q╱M_min, win.Q╱M_hi, q_ad)
end

Row,Limit,Q/M (C/kg),|q_x|
,String,Float64,Float64
1,"Mathieu, lower edge (a₀)",0.829248,0.276507
2,"Gravity, sag = 100 µm",0.867528,0.289271
3,"Adiabatic, max|q| = 0.4",1.19961,0.4
4,"Mathieu, upper edge (b₁)",2.78781,0.929576


Recommended Q/M: 8.68e-01 - 1.20e+00 C/kg


### Parameters maps

In [9]:
# The equations used in the tool are defined in the functions above
plotQ╱MMap(:f, f_lims, Q╱M_lims, η, κ, r_0, z_0, ϵ, V_EC, V_RF, f, N, M;
           drive=drive, e⁻=e⁻, interactive=true, q_ad=q_ad, sag_max=sag_max, g=g,
           V_RF_range=(V_RF_lims[1]:1:V_RF_lims[2]),
           V_EC_range=(V_EC_lims[1]:0.1:V_EC_lims[2]));

In [10]:
plotQ╱MMap(:V_RF, V_RF_lims, Q╱M_lims, η, κ, r_0, z_0, ϵ, V_EC, V_RF, f, N, M;
           drive=drive, e⁻=e⁻, interactive=true, q_ad=q_ad, sag_max=sag_max, g=g,
           f_range=(f_lims[1]:100:f_lims[2])*1e-3,
           V_EC_range=(V_EC_lims[1]:0.1:V_EC_lims[2]));